<a href="https://colab.research.google.com/github/HS-base/FYP/blob/main/Baseline_simulation/CFD_validation_from_experimental_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Strategic Geometric Simplification and Heat Transfer Mapping

To validate the Carnegie-Mellon (C-MU) experimental data using a computationally efficient CFD model, the complex physical geometry of the 240-tube rig is reduced to a simplified 1/4 symmetry smooth-tube unit cell. This requires a rigorous mathematical mapping to relate the physical fluted-tube performance to the idealized smooth-tube CFD baseline.

#### 1. System Scaling and Unit Cell Logic
The physical evaporator contains 240 vertical, doubly-fluted tubes. To minimize computational overhead while preserving the fundamental falling-film fluid dynamics, the CFD domain is restricted to a 1/4 symmetry slice of a single tube.
* **Global Scaling Factor ($K$):** The flow rates from the full-scale experiment are scaled down to the CFD domain using the multiplier $K = 240 \times 4 = 960$.

#### 2. Mapping the Outer Surface Performance (Ammonia Side)
The experimental tubes feature a fluted profile ($r_{peak} = 0.000508\text{ m}$) that drastically enhances evaporative heat transfer. Because simulating these micro-scale flutes in a multiphase CFD framework is numerically prohibitive, the CFD geometry uses a **fluteless, smooth tube**. We must mathematically "strip" the physical enhancement from the experimental data to find the baseline smooth-tube heat transfer coefficient ($h_{out, CFD\_target}$).

This is achieved using the **Analytical Gregorig Multiplier ($F_G$)**, which accounts for two distinct physical enhancements:
* **Macroscopic Enhancement (Area Ratio):** The physical increase in surface area provided by the flutes, calculated from the CAD geometry as $A_{fluted\_raw} / A_{smooth}$.
* **Microscopic Enhancement (Capillary Thinning):** High-pressure zones at the flute crests drive the liquid into the valleys, creating an ultra-thin film at the peaks. This enhancement scales analytically with the 4th root of the macroscopic-to-microscopic radius ratio.



The overall theoretical enhancement is defined as:
$$F_{Gregorig} = Ratio_{area} \times \left( \frac{D_{outer} / 2}{r_{peak}} \right)^{0.25}$$

Once the true experimental ammonia-side coefficient ($h_{out, exp}$) is extracted from the overall thermal resistance network, it is mapped to the smooth-tube CFD geometry by dividing out the Gregorig effect:
$$h_{out, CFD\_target} = \frac{h_{out, exp}}{F_{Gregorig}}$$

#### 3. Mapping the Inner Surface Performance (Water Side)
The internal water side of the C-MU tubes is also fluted, providing an experimentally observed enhancement over standard pipe flow. To map this to the smooth-bore inner wall of the CFD geometry, we build the performance from the ground up using empirical pipe-flow correlations.

* **Smooth-Tube Baseline:** The theoretical internal heat transfer is first calculated using the Dittus-Boelter correlation for cooling fluids:
$$Nu_{w, smooth} = 0.023 \times Re_w^{0.8} \times Pr_w^{0.3}$$

* **Baseline Coefficient:** The smooth-tube coefficient is extracted using the thermal conductivity and internal diameter:
$$h_{in, smooth} = \frac{Nu_{w, smooth} \times k_w}{D_{inner}}$$

* **Applying the Flute Enhancement:** Carnegie-Mellon data specifies a doubly-fluted internal enhancement factor ($E_{in} = 2.5$). The actual enhanced coefficient required for the overall thermal resistance network is computed as:
$$h_{in, enhanced} = h_{in, smooth} \times E_{in}$$

By clearly defining $h_{out, CFD\_target}$ and $h_{in, enhanced}$, we successfully bridge the gap between the high-performance experimental fluted geometry and the mathematically robust smooth-tube CFD unit cell.

In [ ]:
import numpy as np

# ==========================================
# 1. EXACT CAD GEOMETRY
# ==========================================
D_outer = 0.0309372              # [m] Nominal Outer Diameter
L_tube = 4.3688                  # [m] Vertical length
a = 0.0010541                    # [m] Flute Amplitude
p = 0.002025                     # [m] Flute Pitch
A_fluted_raw = 0.6534            # [m2] Total developed area
num_tubes = 240

# CAD Area Ratio
A_smooth = np.pi * D_outer * L_tube
area_ratio = A_fluted_raw / A_smooth

# ==========================================
# 2. ANALYTICAL GREGORIG ENHANCEMENT
# ==========================================
# The radius of curvature at the peak of the flute from reference drawings

r_peak = 0.000508

# The analytical capillary enhancement factor scales with the
# 4th root of the macroscopic to microscopic radius ratio.
capillary_factor = ( (D_outer / 2) / r_peak )**0.25

# Total Theoretical Multiplier
F_G_analytical = area_ratio * capillary_factor

# ==========================================
# 3. RESULTS
# ==========================================
print(f"--- Geometric Properties ---")
print(f"CAD Area Ratio: {area_ratio:.3f}")
print(f"Peak Radius of Curvature (r_peak): {r_peak:.6f} m")

print(f"\n--- Theoretical Enhancement ---")
print(f"Capillary Thinning Factor: {capillary_factor:.3f}")
print(f"Analytical Gregorig Multiplier (F_G): {F_G_analytical:.3f}")

--- Geometric Properties ---
CAD Area Ratio: 1.539
Peak Radius of Curvature (r_peak): 0.000508 m

--- Theoretical Enhancement ---
Capillary Thinning Factor: 2.349
Analytical Gregorig Multiplier (F_G): 3.615


In [ ]:
# Properties of water at 25 °C
mu = 0.87e-3   # dynamic viscosity [Pa·s]
cp = 4182      # specific heat [J/(kg·K)]
k = 0.606      # thermal conductivity [W/(m·K)]

# Prandtl number formula: Pr = (cp * mu) / k
Pr = (cp * mu) / k

print(f"Prandtl number for water at 25 °C: {Pr:.2f}")


Prandtl number for water at 25 °C: 6.00


### Deriving the Target Heat Duty and Acceptability Envelopes

After establishing the geometric mapping and extracting the smooth-tube convective coefficients, the next phase is to calculate the exact, localized heat duty expected for the 1/4 symmetry CFD model. This is achieved by isolating the thermal resistance of the ammonia film and calculating the specific temperature drop across it.

#### 1. The Thermal Resistance Network
The experimental data provides a global overall heat transfer coefficient ($U_O$) for Run 85 of $6240.41\text{ W/m}^2\text{K}$. This global value accounts for three primary thermal resistances in series: the inner water convection ($R_{in}$), the aluminum tube wall conduction ($R_{wall}$), and the outer ammonia film convection ($R_{out}$).

To find the performance of the ammonia film alone, we isolate its resistance from the global network:
$$R_{out} = R_{total} - R_{in} - R_{wall}$$

Using this isolated resistance, the true experimental outer coefficient ($h_{out, exp}$) is extracted and subsequently converted to our smooth-tube CFD target ($h_{out, CFD\_target}$) using the Gregorig multiplier calculated in the previous step.

#### 2. Exact Temperature Profiling (LMTD and Film $\Delta T$)
Heat transfer in the CFD domain is driven by the local temperature gradient. First, the total Log Mean Temperature Difference (LMTD) of the system is calculated using the experimental water inlet ($299.64\text{ K}$) and outlet ($298.54\text{ K}$) temperatures against the constant ammonia saturation temperature ($297.099\text{ K}$).

However, the ammonia film only "feels" a fraction of this total temperature driving force. The specific temperature drop across the ammonia film ($\Delta T_{ammonia}$) is strictly proportional to its share of the total thermal resistance:
$$\Delta T_{ammonia} = LMTD \times \left( \frac{R_{out}}{R_{total}} \right)$$

#### 3. Target Heat Duty Calculation (1/4 Symmetry)
With the idealized smooth-tube convective coefficient and the exact temperature drop isolated, the target heat transfer rate ($Q_{CFD\_target}$) for the localized unit cell is calculated. We multiply these values by the smooth-bore outer surface area of our 1/4 symmetry CFD slice ($A_{CFD\_outer\_smooth}$):
$$Q_{CFD\_target} = h_{out, CFD\_target} \times A_{CFD\_outer\_smooth} \times \Delta T_{ammonia}$$

#### 4. Defining the Acceptability Envelopes
Because multiphase CFD simulations inherently involve numerical diffusion, discretization errors, and modeling assumptions (such as the rigid 1/4 symmetry boundaries), demanding an exact match to the $367.06\text{ W}$ target is numerically unrealistic. Instead, we define strict acceptability envelopes to validate the simulation:
* **High-Fidelity Validation Envelope ($\pm 2\%$):** Represents a highly accurate resolution of the film physics and interfacial heat transfer ($359.72\text{ W}$ to $374.40\text{ W}$).
* **Standard Validation Envelope ($\pm 5\%$):** Represents physically acceptable bounds that account for experimental uncertainties and typical CFD mesh variations ($348.71\text{ W}$ to $385.41\text{ W}$).

In [ ]:
import numpy as np

# ==========================================
# 1. INPUTS & EXPERIMENTAL GROUND TRUTH
# ==========================================
U_O_metric = 6240.41             # [W/m2.K]

# Run 85 Temperatures
T_w_in_exp = 299.64              # [K] Water Inlet
T_w_out_exp = 298.54             # [K] Water Outlet
T_sat = 297.099                  # [K] Ammonia Saturation

# Flow rates
m_dot_water_total = 209.096 # [kg/s]
m_dot_water_tube = m_dot_water_total / num_tubes

# ==========================================
# 2. GEOMETRY
# ==========================================
D_outer = 0.0319532
L_tube = 4.3688
D_inner = D_outer - (2 * 0.00165)
A_out_tube = np.pi * D_outer * L_tube
A_in_tube = np.pi * D_inner * L_tube

A_CFD_outer_smooth = A_out_tube / 4.0

# ==========================================
# 3. INTERNAL FLUID DYNAMICS (Water)
# ==========================================
mu_w = 0.00087
k_w = 0.606
Pr_w = 6.0

Re_w = (4 * m_dot_water_tube) / (np.pi * D_inner * mu_w)

# Dittus-Boelter for cooling fluid (n = 0.3)
n = 0.3
Nu_w_smooth = 0.023 * (Re_w**0.8) * (Pr_w**n)

h_in_smooth = (Nu_w_smooth * k_w) / D_inner

E_in = 2.5 # CMU Doubly-Fluted Internal Enhancement
h_in_enhanced = h_in_smooth * E_in

# ==========================================
# 4. THERMAL RESISTANCES
# ==========================================
R_total = 1.0 / (U_O_metric * A_out_tube)
R_in = 1.0 / (h_in_enhanced * A_in_tube)

k_al = 237.0
R_wall = np.log(D_outer / D_inner) / (2 * np.pi * k_al * L_tube)

# Isolate Ammonia Resistance
R_out = R_total - R_in - R_wall
h_out_exp = 1.0 / (R_out * A_out_tube)
h_out_CFD_target = h_out_exp / F_G_analytical

# ==========================================
# 5. EXACT TEMPERATURE DROPS & TARGET DUTY
# ==========================================
# Calculate LMTD
delta_T1 = T_w_in_exp - T_sat
delta_T2 = T_w_out_exp - T_sat
LMTD = (delta_T1 - delta_T2) / np.log(delta_T1 / delta_T2)

# Calculate Delta T across the ammonia film
delta_T_ammonia = LMTD * (R_out / R_total)

# Calculate final CFD 1/4 symmetry heat duty target
Q_CFD_target = h_out_CFD_target * A_CFD_outer_smooth * delta_T_ammonia

# ==========================================
# 6. RESULTS
# ==========================================
print(f"--- Exact Temperature Profiling ---")
print(f"Total LMTD: {LMTD:.4f} K")
print(f"Ammonia Film Delta T: {delta_T_ammonia:.4f} K")
print(f"Water side Reynold's number: {Re_w :.4f} ")

print(f"\n--- Convective Coefficients ---")
print(f"True Experimental h_out: {h_out_exp:.2f} W/m^2.K")
print(f"Target Smooth CFD h_out: {h_out_CFD_target:.2f} W/m^2.K")

print(f"\n--- FINAL CFD TARGETS ---")
print(f"Target CFD Heat Duty (1/4 symmetry): {Q_CFD_target:.2f} W")
print(f"Acceptability Envelope (-5% to +5%): {Q_CFD_target*0.95:.2f} W  -->  {Q_CFD_target*1.05:.2f} W")
print(f"Acceptability Envelope (-2% to +2%): {Q_CFD_target*0.98:.2f} W  -->  {Q_CFD_target*1.02:.2f} W")

--- Exact Temperature Profiling ---
Total LMTD: 1.9393 K
Ammonia Film Delta T: 0.6113 K
Water side Reynold's number: 44499.2015 

--- Convective Coefficients ---
True Experimental h_out: 19798.04 W/m^2.K
Target Smooth CFD h_out: 5476.96 W/m^2.K

--- FINAL CFD TARGETS ---
Target CFD Heat Duty (1/4 symmetry): 367.06 W
Acceptability Envelope (-5% to +5%): 348.71 W  -->  385.41 W
Acceptability Envelope (-2% to +2%): 359.72 W  -->  374.40 W
